In [1]:
# ============================================================
# PS S6E5 - exp_20260516_033_blend_logreg_topk_feature_variants_min
#
# D: blend-code small improvements
# - Avg / HC / NM / Signed SLSQP on raw OOF predictions
# - Prune weak models
# - Ridge / LogReg on current raw+rank+logit meta features
# - Additional LogReg feature variants: logit_only, raw_logit, raw_only
# - Additional LogReg topK equal blends from C-search candidates
# - Additional Ridge feature variants: logit_only, raw_logit
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260516_033_blend_logreg_topk_feature_variants_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)
RUN_D_EXTRA = False

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================
#
# D experiment stack:
# - Drop 030/031 because shared011 CatBoost meta-stacking did not improve Public LB.
# - Drop old RealMLP variants 007/022/023/027.
# - Drop 018 by default because recent optimized blends effectively zeroed it out after 029/032.
# - Keep 014/015/016/017/021/026/028/029/032.
#
# If you want a conservative comparison, re-add 018 manually and rerun.
# ============================================================

MODEL_SPECS = [
    {
        "key": "014",
        "name": "cat_shared004_no_tyreageratio_str_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "pred": "pred_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "public_lb": 0.95233,
    },
    {
        "key": "015",
        "name": "cat_shared004_no_race_year_groupstats_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "pred": "pred_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "public_lb": 0.95244,
    },
    {
        "key": "016",
        "name": "xgb_shared005_fe_te_seedens_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "pred": "pred_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "017",
        "name": "xgb_shared005_no_pitstop_pairte_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "pred": "pred_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "021",
        "name": "tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min",
        "family": "TabM",
        "oof": "oof_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "pred": "pred_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "public_lb": 0.95171,
    },
    {
        "key": "026",
        "name": "lgb_weather_year_stint_flags_min",
        "family": "LightGBM",
        "oof": "oof_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
        "pred": "pred_exp_20260514_026_lgb_weather_year_stint_flags_min.npy",
        "public_lb": 0.95096,
    },
    {
        "key": "028",
        "name": "parent_child_mlp_shared010_min",
        "family": "Parent-Child MLP",
        "oof": "oof_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "pred": "pred_exp_20260514_028_parent_child_mlp_shared010_min_all_models_mean.npy",
        "public_lb": 0.95260,
    },
    {
        "key": "029",
        "name": "realmlp_shared001v2_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260514_029_realmlp_shared001v2_min.npy",
        "pred": "pred_exp_20260514_029_realmlp_shared001v2_min.npy",
        "public_lb": 0.95397,
    },
    # {
    #     "key": "031",
    #     "name": "catboost_meta_stacking_shared011_tawara_gpu_min",
    #     "family": "CatBoost",
    #     "oof": "oof_exp_20260514_031_catboost_meta_stacking_shared011_tawara_gpu_min.npy",
    #     "pred": "pred_exp_20260514_031_catboost_meta_stacking_shared011_tawara_gpu_min.npy",
    #     "public_lb": 0.95220,
    # },
    {
        "key": "032",
        "name": "custom_realmlp_cv095352_min",
        "family": "CustomTorchRealMLP",
        "oof": "oof_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "pred": "pred_exp_20260516_032_custom_realmlp_cv095352_min.npy",
        "public_lb": 0.95309,
    },
    # {
    #     "key": "034",
    #     "name": "lgb_laptime_delta_nonlinear_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260517_034_lgb_laptime_delta_nonlinear_min.npy",
    #     "pred": "pred_exp_20260517_034_lgb_laptime_delta_nonlinear_min.npy",
    #     "public_lb": 0.95089,
    # },
    # {
    #     "key": "035",
    #     "name": "xgb_laptime_delta_nonlinear_min",
    #     "family": "XGBoost",
    #     "oof": "oof_exp_20260518_035_xgb_laptime_delta_nonlinear_min.npy",
    #     "pred": "pred_exp_20260518_035_xgb_laptime_delta_nonlinear_min.npy",
    #     "public_lb": 0.95162,
    # },
    {
        "key": "036",
        "name": "realmlp_shared001v2_no_race_year_te_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260518_036_realmlp_shared001v2_no_race_year_te_min.npy",
        "pred": "pred_exp_20260518_036_realmlp_shared001v2_no_race_year_te_min.npy",
        "public_lb": 0.95393,
    },
    {
        "key": "037",
        "name": "realmlp_shared001v2_pitstop_light_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.npy",
        "pred": "pred_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.npy",
        "public_lb": 0.95394,
    },
    {
        "key": "038",
        "name": "realmlp_shared001v2_seed_reg_variant_min",
        "family": "RealMLP",
        "oof":   "oof_exp_20260518_038_realmlp_shared001v2_seed_reg_variant_min.npy",
        "pred": "pred_exp_20260518_038_realmlp_shared001v2_seed_reg_variant_min.npy",
        "public_lb": 0.95384,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("model_keys:", model_keys)
print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)


model_keys: ['014', '015', '016', '017', '021', '026', '028', '029', '032', '036', '037', '038']
X_raw: (439140, 12)
T_raw: (188165, 12)


In [7]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
9,036,realmlp_shared001v2_no_race_year_te_min,RealMLP,0.954075,0.95393,3.469280e-06,0.999499,0.000007,0.999095
11,038,realmlp_shared001v2_seed_reg_variant_min,RealMLP,0.954065,0.95384,6.162125e-06,0.999215,0.000007,0.999218
7,029,realmlp_shared001v2_min,RealMLP,0.954042,0.95397,4.969082e-08,0.998981,0.000008,0.998439
10,037,realmlp_shared001v2_pitstop_light_min,RealMLP,0.954013,0.95394,1.225402e-06,0.998859,0.000010,0.998104
8,032,custom_realmlp_cv095352_min,CustomTorchRealMLP,0.953506,0.95309,3.416236e-04,0.998800,0.000474,0.998323
0,014,cat_shared004_no_tyreageratio_str_min,CatBoost,0.952726,0.95233,7.059056e-05,0.997215,0.000153,0.996355
1,015,cat_shared004_no_race_year_groupstats_min,CatBoost,0.952714,0.95244,4.489366e-05,0.997058,0.000141,0.996636
2,016,xgb_shared005_fe_te_seedens_min,XGBoost,0.951986,0.95164,1.674493e-05,0.997698,0.000027,0.997298
3,017,xgb_shared005_no_pitstop_pairte_min,XGBoost,0.951939,0.95164,1.331864e-05,0.997702,0.000024,0.997621
4,021,tabm_shared007_comp_oof_no_pitstop_no_isorigin...,TabM,0.951492,0.95171,1.000000e-06,0.999431,0.000003,0.996751


best single: 036 0.9540748794531666


In [8]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,014,015,016,017,021,026,028,029,032,036,037,038
014,1.000000,0.996917,0.962898,0.962739,0.968765,0.984963,0.956846,0.961507,0.960927,0.960041,0.961575,0.958806
015,0.996917,1.000000,0.962398,0.962237,0.967914,0.984951,0.956020,0.960397,0.959915,0.958872,0.960469,0.957678
016,0.962898,0.962398,1.000000,0.998485,0.978284,0.960326,0.972366,0.980101,0.979115,0.979401,0.980109,0.978208
017,0.962739,0.962237,0.998485,1.000000,0.978150,0.960937,0.972213,0.979812,0.979023,0.979089,0.979927,0.977876
021,0.968765,0.967914,0.978284,0.978150,1.000000,0.966370,0.974491,0.980413,0.982081,0.979649,0.980458,0.979823
026,0.984963,0.984951,0.960326,0.960937,0.966370,1.000000,0.954081,0.957061,0.958169,0.955784,0.957270,0.955488
028,0.956846,0.956020,0.972366,0.972213,0.974491,0.954081,1.000000,0.981320,0.979124,0.980752,0.981278,0.981041
029,0.961507,0.960397,0.980101,0.979812,0.980413,0.957061,0.981320,1.000000,0.990775,0.997876,0.998364,0.995398
032,0.960927,0.959915,0.979115,0.979023,0.982081,0.958169,0.979124,0.990775,1.000000,0.990511,0.990845,0.990752
036,0.960041,0.958872,0.979401,0.979089,0.979649,0.955784,0.980752,0.997876,0.990511,1.000000,0.997881,0.995585


Spearman correlation


,014,015,016,017,021,026,028,029,032,036,037,038
014,1.000000,0.994676,0.975612,0.975146,0.970041,0.973115,0.968397,0.976608,0.972390,0.976233,0.976560,0.974665
015,0.994676,1.000000,0.975249,0.974943,0.969630,0.972735,0.968181,0.976407,0.972607,0.975995,0.976357,0.974520
016,0.975612,0.975249,1.000000,0.998203,0.965983,0.969232,0.967417,0.972249,0.966370,0.971731,0.972120,0.970094
017,0.975146,0.974943,0.998203,1.000000,0.965901,0.969808,0.967124,0.972103,0.966202,0.971444,0.972050,0.969782
021,0.970041,0.969630,0.965983,0.965901,1.000000,0.962785,0.967054,0.967894,0.966383,0.968109,0.967979,0.968751
026,0.973115,0.972735,0.969232,0.969808,0.962785,1.000000,0.959946,0.966439,0.965142,0.966782,0.966543,0.966891
028,0.968397,0.968181,0.967417,0.967124,0.967054,0.959946,1.000000,0.973048,0.967367,0.972625,0.973049,0.973566
029,0.976608,0.976407,0.972249,0.972103,0.967894,0.966439,0.973048,1.000000,0.980598,0.995480,0.996785,0.991009
032,0.972390,0.972607,0.966370,0.966202,0.966383,0.965142,0.967367,0.980598,1.000000,0.981078,0.980526,0.981694
036,0.976233,0.975995,0.971731,0.971444,0.968109,0.966782,0.972625,0.995480,0.981078,1.000000,0.995540,0.992006


In [9]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T, keys):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in keys] +
        [f"rank_{k}" for k in keys] +
        [f"logit_{k}" for k in keys]
    )

    return X_meta, T_meta, feature_names


X_meta_full, T_meta_full, meta_feature_names_full = build_meta_features(
    X_raw,
    T_raw,
    model_keys,
)

print("X_meta_full:", X_meta_full.shape)
print("T_meta_full:", T_meta_full.shape)
print("meta meta_feature_names_full:", meta_feature_names_full)

def build_meta_features_variant(X, T, keys, variant="raw_rank_logit"):
    """Build meta features with explicit feature-set variant.

    Variants:
      - raw_rank_logit: current baseline
      - raw_logit
      - logit_only
      - raw_only
      - rank_only
      - raw_rank
      - rank_logit
    """
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    parts_X = []
    parts_T = []
    names = []

    def add_part(prefix, X_part, T_part):
        parts_X.append(X_part)
        parts_T.append(T_part)
        names.extend([f"{prefix}_{k}" for k in keys])

    if variant == "raw_rank_logit":
        add_part("raw", X, T)
        add_part("rank", X_rank, T_rank)
        add_part("logit", X_logit, T_logit)
    elif variant == "raw_logit":
        add_part("raw", X, T)
        add_part("logit", X_logit, T_logit)
    elif variant == "logit_only":
        add_part("logit", X_logit, T_logit)
    elif variant == "raw_only":
        add_part("raw", X, T)
    elif variant == "rank_only":
        add_part("rank", X_rank, T_rank)
    elif variant == "raw_rank":
        add_part("raw", X, T)
        add_part("rank", X_rank, T_rank)
    elif variant == "rank_logit":
        add_part("rank", X_rank, T_rank)
        add_part("logit", X_logit, T_logit)
    else:
        raise ValueError(f"Unknown meta feature variant: {variant}")

    return np.column_stack(parts_X), np.column_stack(parts_T), names


X_meta_full: (439140, 36)
T_meta_full: (188165, 36)
meta meta_feature_names_full: ['raw_014', 'raw_015', 'raw_016', 'raw_017', 'raw_021', 'raw_026', 'raw_028', 'raw_029', 'raw_032', 'raw_036', 'raw_037', 'raw_038', 'rank_014', 'rank_015', 'rank_016', 'rank_017', 'rank_021', 'rank_026', 'rank_028', 'rank_029', 'rank_032', 'rank_036', 'rank_037', 'rank_038', 'logit_014', 'logit_015', 'logit_016', 'logit_017', 'logit_021', 'logit_026', 'logit_028', 'logit_029', 'logit_032', 'logit_036', 'logit_037', 'logit_038']


In [10]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
    weight_keys=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "weight_keys": None if weights is None else list(weight_keys or model_keys),
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        f"delta_vs_{best_single_key}": float(score - auc(y, oofs[best_single_key])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [11]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    # "avg_007_014_015_016_018_020_021_022_023": ["007", "014", "015", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_016_018_020_021_022_023": ["007", "014", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_015_018_020_021_022_023": ["007", "014", "015", "018", "020", "021", "022", "023"],
    "avg" : model_keys
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg: 0.954796880


In [12]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.954908109
HC weights:
014 0.054964891726606006
015 0.04959034298732811
016 0.1328399722455249
017 0.004891528061835402
021 0.028733204471671554
026 0.04959033500317643
028 0.04959432706304901
029 0.07603785235368847
032 0.12643872646436452
036 0.18166253697230858
037 0.0
038 0.24565628265044712


In [13]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.954908
         Iterations: 256
         Function evaluations: 618
nm_softmax_raw: 0.954908151
NM weights:
014 0.05384018736913262
015 0.05014439412701098
016 0.13239244544876652
017 0.005108984840696811
021 0.028725633632565334
026 0.04977324059132917
028 0.049841066012853
029 0.07662869727033468
032 0.12636243449395956
036 0.18148740428827523
037 6.332134906346043e-09
038 0.2456955055929413


In [14]:
# ============================================================
# Signed SLSQP weights
# Allow small negative weights, with sum(w)=1
# ============================================================

def weighted_average_signed(X, w):
    return np.asarray(X, dtype=float) @ np.asarray(w, dtype=float)


def optimize_signed_slsqp(
    X,
    y,
    start_w=None,
    neg_bound=-0.10,
    pos_bound=0.60,
    maxiter=1000,
):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n
    else:
        start_w = np.asarray(start_w, dtype=float)

    # Start must satisfy sum=1
    start_w = start_w / start_w.sum()

    bounds = [(neg_bound, pos_bound) for _ in range(n)]

    constraints = [
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1.0,
        }
    ]

    def objective(w):
        p = weighted_average_signed(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        start_w,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": maxiter,
            "ftol": 1e-12,
            "disp": True,
        },
    )

    w = res.x
    score = auc(y, weighted_average_signed(X, w))

    return w, score, res

In [15]:
print("\n" + "=" * 80)
print("Signed SLSQP")
print("=" * 80)

signed_w, signed_score, signed_res = optimize_signed_slsqp(
    X_raw,
    y,
    start_w=nm_w,
    neg_bound=-0.05,
    pos_bound=0.50,
    maxiter=1000,
)

add_candidate(
    name="slsqp_signed_raw_neg005",
    method="slsqp_signed",
    oof_pred=weighted_average_signed(X_raw, signed_w),
    test_pred=weighted_average_signed(T_raw, signed_w),
    weights=signed_w,
    params={
        "constraint": "sum1_signed",
        "neg_bound": -0.05,
        "pos_bound": 0.50,
        "success": bool(signed_res.success),
        "message": str(signed_res.message),
        "nit": int(getattr(signed_res, "nit", -1)),
        "fun": float(signed_res.fun),
    },
    notes="signed SLSQP on raw OOF predictions; high overfit risk; attack-only",
)

print("Signed SLSQP weights:")
for k, w in zip(model_keys, signed_w):
    print(k, w)

print("Signed score:", signed_score)
print("sum weights:", signed_w.sum())
print("min weight:", signed_w.min())
print("max weight:", signed_w.max())


Signed SLSQP
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.954908109605415
            Iterations: 20
            Function evaluations: 451
            Gradient evaluations: 20
slsqp_signed_raw_neg005: 0.954908110
Signed SLSQP weights:
014 0.05393240356600132
015 0.05011701107861439
016 0.13242686360474643
017 0.005345581657354534
021 0.02849588160860323
026 0.050000705916371437
028 0.04938461229987873
029 0.076742751027384
032 0.12623906044818495
036 0.18156335782199237
037 1.8341696348040522e-05
038 0.2457334292745208
Signed score: 0.954908109605415
sum weights: 1.0000000000000004
min weight: 1.8341696348040522e-05
max weight: 0.2457334292745208


In [16]:
# ============================================================
# Prune models based on HC / NM / SLSQP weights
# ============================================================

PRUNE_HARD_THRESHOLD = 0.005
PRUNE_SOFT_THRESHOLD = 0.020
KEEP_PROTECTED_KEYS = {
    "014", "015",  # CatBoost branch
                   # XGB branch; 017 is historically useful for LogReg "016", "017",
    "021",          # TabM branch
    "034",          # LGB branch
    "028",          # structurally different NN branch
    "029",          # current main RealMLP representative
    "032",          # custom Torch RealMLP auxiliary branch
}

def build_pruned_model_keys(model_keys, hc_w, nm_w, signed_w):
    rows = []
    for k, wh, wn, ws in zip(model_keys, hc_w, nm_w, signed_w):
        max_w = max(float(wh), float(wn), float(ws))
        min_w = min(float(wh), float(wn), float(ws))

        if max_w < PRUNE_HARD_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_hard"
        elif max_w < PRUNE_SOFT_THRESHOLD and k not in KEEP_PROTECTED_KEYS:
            decision = "drop_soft"
        else:
            decision = "keep"

        rows.append({
            "key": k,
            "hc_weight": float(wh),
            "nm_weight": float(wn),
            "slsqp_weight": float(ws),
            "max_weight": max_w,
            "min_weight": min_w,
            "protected": k in KEEP_PROTECTED_KEYS,
            "decision": decision,
        })

    prune_df = pd.DataFrame(rows)
    keep_keys = prune_df.loc[prune_df["decision"] == "keep", "key"].tolist()
    drop_keys = prune_df.loc[prune_df["decision"] != "keep", "key"].tolist()

    return keep_keys, drop_keys, prune_df


keep_keys, drop_keys, prune_df = build_pruned_model_keys(
    model_keys=model_keys,
    hc_w=hc_w,
    nm_w=nm_w,
    signed_w=signed_w,
)

print("keep_keys:", keep_keys)
print("drop_keys:", drop_keys)
display(prune_df)

prune_df.to_csv(CFG.OUTDIR / "prune_decision_after_hc_nm_slsqp.csv", index=False)

keep_keys: ['014', '015', '016', '021', '026', '028', '029', '032', '036', '038']
drop_keys: ['017', '037']


,key,hc_weight,nm_weight,slsqp_weight,max_weight,min_weight,protected,decision
0,014,0.054965,5.384019e-02,0.053932,0.054965,0.053840,True,keep
1,015,0.049590,5.014439e-02,0.050117,0.050144,0.049590,True,keep
2,016,0.132840,1.323924e-01,0.132427,0.132840,0.132392,False,keep
3,017,0.004892,5.108985e-03,0.005346,0.005346,0.004892,False,drop_soft
4,021,0.028733,2.872563e-02,0.028496,0.028733,0.028496,True,keep
5,026,0.049590,4.977324e-02,0.050001,0.050001,0.049590,False,keep
6,028,0.049594,4.984107e-02,0.049385,0.049841,0.049385,True,keep
7,029,0.076038,7.662870e-02,0.076743,0.076743,0.076038,True,keep
8,032,0.126439,1.263624e-01,0.126239,0.126439,0.126239,True,keep
9,036,0.181663,1.814874e-01,0.181563,0.181663,0.181487,False,keep


In [17]:
# ============================================================
# Rebuild matrices for pruned Ridge / LogReg
# ============================================================

pruned_idx = [model_keys.index(k) for k in keep_keys]

model_keys_full = model_keys.copy()
model_keys_pruned = keep_keys.copy()

X_raw_full = X_raw
T_raw_full = T_raw

X_raw_pruned = X_raw[:, pruned_idx]
T_raw_pruned = T_raw[:, pruned_idx]

# Temporarily switch global model_keys for meta feature names
model_keys = model_keys_pruned
X_meta, T_meta, meta_feature_names = build_meta_features(
    X_raw_pruned,
    T_raw_pruned,
    keep_keys,
)

print("Pruned X_raw:", X_raw_pruned.shape)
print("Pruned X_meta:", X_meta.shape)
print("Pruned model_keys:", model_keys)

Pruned X_raw: (439140, 10)
Pruned X_meta: (439140, 30)
Pruned model_keys: ['014', '015', '016', '021', '026', '028', '029', '032', '036', '038']


In [18]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_pruned",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-4, 1e-1, 7)
    for l1 in [0.05, 0.10, 0.20, 0.50, 0.90]
]


def build_elastic_refined_grid_fast(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.5,
        factor_high=2.0,
        n=7,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = sorted(set([
        round(max(0.001, l1_center - 0.10), 6),
        round(l1_center, 6),
        round(min(0.999, l1_center + 0.10), 6),
        0.05,
        0.10,
        0.20,
        0.50,
        0.90,
    ]))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


# elastic_best, elastic_hist = run_meta_regressor_two_stage(
#     name="elasticnet_meta_all",
#     estimator_factory=lambda p: ElasticNet(
#         alpha=p["alpha"],
#         l1_ratio=p["l1_ratio"],
#         max_iter=30000,
#         random_state=CFG.SEED,
#         selection="cyclic",
#     ),
#     coarse_grid=elastic_coarse_grid,
#     refine_builder=build_elastic_refined_grid_fast,
# )


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_pruned",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_pruned coarse search
{'alpha': np.float64(0.001)} 0.954725668856546
{'alpha': np.float64(0.0021544346900318843)} 0.9547256691493522
{'alpha': np.float64(0.004641588833612777)} 0.9547256704832472
{'alpha': np.float64(0.01)} 0.9547256714918019
{'alpha': np.float64(0.021544346900318832)} 0.9547256753633506
{'alpha': np.float64(0.046415888336127774)} 0.954725678649287
{'alpha': np.float64(0.1)} 0.9547256912724881
{'alpha': np.float64(0.21544346900318823)} 0.9547257184383973
{'alpha': np.float64(0.46415888336127775)} 0.9547257854910189
{'alpha': np.float64(1.0)} 0.9547259246065021
{'alpha': np.float64(2.154434690031882)} 0.9547261927193871
{'alpha': np.float64(4.6415888336127775)} 0.9547267193801539
{'alpha': np.float64(10.0)} 0.9547277266009768
{'alpha': np.float64(21.54434690031882)} 0.9547294918645395
{'alpha': np.float64(46.41588833612773)} 0.9547317450083135
{'alpha': np.float64(100.0)} 0.9547328085780646
{'alpha': np.float64(21

,stage,score,alpha
28,refined,0.954733,85.133992
15,coarse,0.954733,100.000000
29,refined,0.954733,100.000000
27,refined,0.954733,72.477966
30,refined,0.954732,117.461894
26,refined,0.954732,61.703386
25,refined,0.954732,52.530556
31,refined,0.954732,137.972966
14,coarse,0.954732,46.415888
24,refined,0.954732,44.721360



logreg_meta_pruned coarse search
{'C': np.float64(0.001)} 0.9548641759208116
{'C': np.float64(0.0021544346900318843)} 0.9548689801874777
{'C': np.float64(0.004641588833612777)} 0.9548707217988055
{'C': np.float64(0.01)} 0.9548658656729065
{'C': np.float64(0.021544346900318832)} 0.9548675137163838
{'C': np.float64(0.046415888336127774)} 0.9548671016729804
{'C': np.float64(0.1)} 0.9548705538256439
{'C': np.float64(0.21544346900318823)} 0.9548683619759696
{'C': np.float64(0.46415888336127775)} 0.9548688096766624
{'C': np.float64(1.0)} 0.9548704411277877
{'C': np.float64(2.154434690031882)} 0.9548657565537928
{'C': np.float64(4.6415888336127775)} 0.9548654284481694
{'C': np.float64(10.0)} 0.9548660780550098
{'C': np.float64(21.54434690031882)} 0.9548649559890849
{'C': np.float64(46.41588833612773)} 0.9548666716707963
{'C': np.float64(100.0)} 0.9548673939586446
{'C': np.float64(215.44346900318823)} 0.954866221855392
{'C': np.float64(464.1588833612773)} 0.9548661847015376
{'C': np.float64(1

,stage,score,C
2,coarse,0.954871,0.004642
29,refined,0.954871,0.004642
6,coarse,0.954871,0.100000
9,coarse,0.954870,1.000000
28,refined,0.954870,0.003952
26,refined,0.954870,0.002864
33,refined,0.954870,0.008836
32,refined,0.954870,0.007522
25,refined,0.954869,0.002438
30,refined,0.954869,0.005452


In [19]:
# ============================================================
# D additions: LogReg topK and feature-set variants
# ============================================================
#
# Purpose:
# - Absorb only the low-cost ideas from stacking_vibe_coding.
# - Do not introduce full multi-layer stacking.
#
# Added candidates:
# - logreg_meta_pruned_top3_equal / top5_equal from raw+rank+logit C-search
# - logreg variants: logit_only, raw_logit, raw_only
# - ridge variants: logit_only, raw_logit
# ============================================================

def fit_predict_logreg_meta_features(X_feat, T_feat, C):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_feat[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_feat[va_idx])[:, 1]

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_feat, y)
    test_meta = final_model.predict_proba(T_feat)[:, 1]

    return oof_meta, test_meta, auc(y, oof_meta)


def fit_predict_ridge_meta_features(X_feat, T_feat, alpha):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            Ridge(
                alpha=alpha,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_feat[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_feat[va_idx])

    final_model = make_pipeline(
        StandardScaler(),
        Ridge(
            alpha=alpha,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_feat, y)
    test_meta = final_model.predict(T_feat)

    return oof_meta, test_meta, auc(y, oof_meta)


def run_logreg_two_stage_on_feature_variant(variant):
    X_v, T_v, names_v = build_meta_features_variant(
        X_raw_pruned,
        T_raw_pruned,
        keep_keys,
        variant=variant,
    )

    print(f"\nLogReg feature variant: {variant}")
    print("X_v:", X_v.shape)

    coarse_grid = [{"C": c} for c in np.geomspace(1e-3, 1e3, 19)]

    history = []
    best = None

    for params in coarse_grid:
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_v, T_v, params["C"])
        history.append({"stage": "coarse", "score": float(score), **params})
        print("coarse", variant, params, score)
        if best is None or score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    refined_grid = [
        {"C": c}
        for c in make_refined_grid_1d(
            best["params"]["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ]

    for params in refined_grid:
        oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_v, T_v, params["C"])
        history.append({"stage": "refined", "score": float(score), **params})
        print("refined", variant, params, score)
        if score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    name = f"logreg_meta_pruned_{variant}"
    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=best["pred"],
        params={**best["params"], "feature_variant": variant},
        notes=f"D feature-set variant: LogisticRegression on {variant} meta features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)
    display(hist_df.head(20))

    return best, hist_df


def run_ridge_two_stage_on_feature_variant(variant):
    X_v, T_v, names_v = build_meta_features_variant(
        X_raw_pruned,
        T_raw_pruned,
        keep_keys,
        variant=variant,
    )

    print(f"\nRidge feature variant: {variant}")
    print("X_v:", X_v.shape)

    coarse_grid = [{"alpha": a} for a in np.geomspace(1e-3, 1e3, 19)]

    history = []
    best = None

    for params in coarse_grid:
        oof_meta, test_meta, score = fit_predict_ridge_meta_features(X_v, T_v, params["alpha"])
        history.append({"stage": "coarse", "score": float(score), **params})
        print("coarse", variant, params, score)
        if best is None or score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    refined_grid = [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best["params"]["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ]

    for params in refined_grid:
        oof_meta, test_meta, score = fit_predict_ridge_meta_features(X_v, T_v, params["alpha"])
        history.append({"stage": "refined", "score": float(score), **params})
        print("refined", variant, params, score)
        if score > best["score"]:
            best = {"score": score, "params": params, "oof": oof_meta, "pred": test_meta}

    name = f"ridge_meta_pruned_{variant}"
    add_candidate(
        name=name,
        method="ridge",
        oof_pred=best["oof"],
        test_pred=best["pred"],
        params={**best["params"], "feature_variant": variant},
        notes=f"D feature-set variant: Ridge on {variant} meta features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)
    display(hist_df.head(20))

    return best, hist_df

if RUN_D_EXTRA:
    # logreg topK / feature variants / ridge variants
    print("\n" + "=" * 80)
    print("D additions: LogReg topK and feature-set variants")
    print("=" * 80)


    # ----------------------------
    # LogReg topK equal blend using the baseline raw+rank+logit search history
    # ----------------------------
    
    def add_logreg_topk_equal_from_history(k):
        hist = logreg_hist.sort_values("score", ascending=False).copy()
        top = hist.drop_duplicates(subset=["C"]).head(k)
    
        oofs_top = []
        preds_top = []
        rows = []
    
        for _, row in top.iterrows():
            C = float(row["C"])
            oof_meta, test_meta, score = fit_predict_logreg_meta_features(X_meta, T_meta, C)
            oofs_top.append(oof_meta)
            preds_top.append(test_meta)
            rows.append({"C": C, "cv_auc_recomputed": float(score), "source_score": float(row["score"])})
    
        oof_avg = np.mean(oofs_top, axis=0)
        pred_avg = np.mean(preds_top, axis=0)
    
        name = f"logreg_meta_pruned_top{k}_equal"
        add_candidate(
            name=name,
            method="logreg_topk_equal",
            oof_pred=oof_avg,
            test_pred=pred_avg,
            params={
                "k": k,
                "feature_variant": "raw_rank_logit",
                "Cs": [float(r["C"]) for r in rows],
            },
            notes=f"D top{k} equal blend of LogisticRegression C candidates on raw+rank+logit features",
        )
    
        top_df = pd.DataFrame(rows)
        top_df.to_csv(CFG.OUTDIR / f"{name}_components.csv", index=False)
        display(top_df)
    
        return top_df
    
    
    logreg_top3_components = add_logreg_topk_equal_from_history(3)
    logreg_top5_components = add_logreg_topk_equal_from_history(5)
    
    
    # ----------------------------
    # LogReg feature-set variants
    # ----------------------------
    
    logreg_variant_results = {}
    for variant in ["logit_only", "raw_logit", "raw_only"]:
        best_v, hist_v = run_logreg_two_stage_on_feature_variant(variant)
        logreg_variant_results[variant] = {
            "best": best_v,
            "history": hist_v,
        }
    
    
    # ----------------------------
    # Ridge feature-set variants
    # ----------------------------
    
    ridge_variant_results = {}
    for variant in ["logit_only", "raw_logit"]:
        best_v, hist_v = run_ridge_two_stage_on_feature_variant(variant)
        ridge_variant_results[variant] = {
            "best": best_v,
            "history": hist_v,
        }


In [20]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    weight_keys = rec.get("weight_keys") or model_keys
    for k, w in zip(weight_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_036,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
2,nm_softmax_raw,nm,0.954908,0.000833,0.000833,"[0.05384019, 0.05014439, 0.13239245, 0.0051089...","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000132,0.996287,0.000151,0.997133
3,slsqp_signed_raw_neg005,slsqp_signed,0.954908,0.000833,0.000833,"[0.0539324, 0.05011701, 0.13242686, 0.00534558...","{""constraint"": ""sum1_signed"", ""neg_bound"": -0....",signed SLSQP on raw OOF predictions; high over...,0.000132,0.996295,0.000150,0.997136
1,hc_nonnegative_raw,hc,0.954908,0.000833,0.000833,"[0.05496489, 0.04959034, 0.13283997, 0.0048915...","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000132,0.996287,0.000151,0.997133
5,logreg_meta_pruned,logreg,0.954871,0.000796,0.000796,None,"{""C"": 0.004641588833612777}",two-stage meta CV logistic regression on raw+r...,0.000042,0.988656,0.000042,0.990147
0,avg,avg,0.954797,0.000722,0.000722,"[0.08333333, 0.08333333, 0.08333333, 0.0833333...",{},"simple average of ['014', '015', '016', '017',...",0.000125,0.995714,0.000157,0.996574
4,ridge_meta_pruned,ridge,0.954733,0.000658,0.000658,None,"{""alpha"": 85.1339922520785}",two-stage meta CV on raw+rank+logit features,-0.065685,0.998211,-0.041078,0.994387


,candidate,method,cv_auc,w_014,w_015,w_016,w_017,w_021,w_026,w_028,w_029,w_032,w_036,w_037,w_038
2,nm_softmax_raw,nm,0.954908,0.053840,0.050144,0.132392,0.005109,0.028726,0.049773,0.049841,0.076629,0.126362,0.181487,6.332135e-09,0.245696
3,slsqp_signed_raw_neg005,slsqp_signed,0.954908,0.053932,0.050117,0.132427,0.005346,0.028496,0.050001,0.049385,0.076743,0.126239,0.181563,1.834170e-05,0.245733
1,hc_nonnegative_raw,hc,0.954908,0.054965,0.049590,0.132840,0.004892,0.028733,0.049590,0.049594,0.076038,0.126439,0.181663,0.000000e+00,0.245656
0,avg,avg,0.954797,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,8.333333e-02,0.083333


In [21]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"sub_{safe_name}_{CFG.EXP_ID}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg 0.9547968803349937 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_avg_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
hc_nonnegative_raw 0.9549081093451427 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_hc_nonnegative_raw_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
nm_softmax_raw 0.9549081509886924 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_nm_softmax_raw_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
slsqp_signed_raw_neg005 0.954908109605415 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_slsqp_signed_raw_neg005_exp_20260516_033_blend_logreg_topk_feature_variants_min.csv
ridge_meta_pruned 0.9547328821374911 /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min/submissions/sub_ridge_meta_pruned_exp_20260516_033_blend_logreg_topk_feature_v

In [22]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-16",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys_full": model_keys_full if "model_keys_full" in globals() else model_keys,
        "model_keys_pruned": model_keys_pruned if "model_keys_pruned" in globals() else model_keys,
        "model_keys_current": model_keys,
        "n_models_full": len(model_keys_full) if "model_keys_full" in globals() else len(model_keys),
        "n_models_pruned": len(model_keys_pruned) if "model_keys_pruned" in globals() else len(model_keys),
        "D_additions": [
            "logreg_meta_pruned_top3_equal",
            "logreg_meta_pruned_top5_equal",
            "logreg_meta_pruned_logit_only",
            "logreg_meta_pruned_raw_logit",
            "logreg_meta_pruned_raw_only",
            "ridge_meta_pruned_logit_only",
            "ridge_meta_pruned_raw_logit"
        ],
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "D experiment: low-cost blend-code improvements only.",
        "Avg/HC/NM/SLSQP are raw full-stack methods before pruning.",
        "Ridge/LogReg baseline and variants are run on the pruned stack.",
        "Added LogReg top3/top5 equal blends and feature-set variants: logit_only, raw_logit, raw_only.",
        "Added Ridge feature-set variants: logit_only, raw_logit."
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260516_033_blend_logreg_topk_feature_variants_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.9549081509886924, 'delta_vs_036': 0.0008332715355258147, 'delta_vs_best_single': 0.0008332715355258147, 'weights': '[0.05384019, 0.05014439, 0.13239245, 0.00510898, 0.02872563, 0.04977324, 0.04984107, 0.0766287, 0.12636243, 0.1814874, 1e-08, 0.24569551]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 256, "fun": -0.9549081509886924}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 0.00013179444401520372, 'oof_max': 0.9962870731028929, 'pred_min': 0.00015060327593478535, 'pred_max': 0.9971334511191113}
